# Avito DS Bootcamp
EDA в рамках тестового задания для Avito DS Bootcamp




---



In [ ]:
!wget -q --show-progress --no-check-certificate 'https://docs.google.com/uc?export=download&id=1r0mVIQfg3HYJKFIDcHwInuBPWsUBon8z' -O events.csv.gz
!wget -q --show-progress --no-check-certificate 'https://docs.google.com/uc?export=download&id=13JonFyBxaGSZJ5F_08D6tYP2uDnv8j1a' -O test.csv
!wget -q --show-progress --no-check-certificate 'https://docs.google.com/uc?export=download&id=1rwsOr06_dM4pupnO1FhOcvqWlHAM0iVe' -O train.csv
!wget -q --show-progress --no-check-certificate 'https://docs.google.com/uc?export=download&id=1DVrGiA2Nh8KvYe4os_rR2sv9ElFFgCAv' -O metric.py

events.csv.gz       100%[===================>]  11.39M  45.4MB/s    in 0.3s    
test.csv            100%[===================>] 297.28K  --.-KB/s    in 0.02s   
train.csv           100%[===================>] 693.25K  --.-KB/s    in 0.03s   
metric.py           100%[===================>]   2.37K  --.-KB/s    in 0s      


In [ ]:
import plotly.express as px

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv('/content/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('/content/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('/content/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [ ]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


## Осмотреться

Прежде чем считать агрегаты, стоит посмотреть на данные: какие типы событий бывают, что
лежит в `platform` и `user_agent`, где пропуски, всё ли уникально.

In [ ]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


## Окно наблюдения

Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [ ]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


In [ ]:
train.shape

(11091, 5)



---
### Попадает ли captcha_shown в окно наблюдения


Хочется разобраться, почему нет типа события `captcha_shown` в тренировочной выборке, и не попадает ли оно в тестовую

In [ ]:
ev = events.merge(train[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
ev[ev['event_name'] == 'captcha_shown'].head(5)

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts
3,ck_6e98fe733ea437fa,2026-04-10 00:02:49,900,captcha_shown,Web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-09,2026-04-10
35,ck_870830faf509a2d9,2026-04-10 06:23:48,900,captcha_shown,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,2026-04-09,2026-04-10
86,ck_f9402f8383819833,2026-04-10 00:02:17,900,captcha_shown,android,Avito/122.0 (Android 13; SM-A515F) okhttp/4.11.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-09,2026-04-10
91,ck_d0b95e9fcb865b39,2026-04-17 00:54:59,900,captcha_shown,Android,Mozilla/5.0 (Linux; Android 12; Redmi Note 11)...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-16,2026-04-17
183,ck_5c92994c35010155,2026-04-12 04:14:57,900,captcha_shown,Web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-11,2026-04-12


In [ ]:
df_check_1 = ev_te.merge(train[['cookie_id', 'target']], on=['cookie_id'], how='left')
df_check_1[df_check_1['event_name'] == 'captcha_shown']

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts,target


Событие `captcha_shown` не попадает в условие фильтрации ни на тестовой, ни на тренировочной выборке, поэтому не учитываем его при анализе

## Куда копать дальше

Подсказок по конкретным признакам не будет — это и есть содержание задания. Несколько
вопросов, которые стоит себе задать:

* чем поток событий робота отличается от потока событий человека, если смотреть не на
  количество, а на **моменты времени**;
* что полезного лежит в строке `user_agent` и почему её нельзя брать как есть;
* насколько разнообразно то, что смотрит кука: объявления, категории, запросы, страницы выдачи;
* всё ли в порядке с самим файлом событий — порядок строк, дубликаты, пропуски;
* какие признаки бесполезны, потому что описывают технические характеристики, а не поведение.

# EDA

In [ ]:
df_check = ev_tr.merge(train[['cookie_id', 'target']], on=['cookie_id'], how='left')

In [ ]:
df_check.shape

(198436, 17)

приведем данные о платформе к единому формату и заменим iphone на ios

In [ ]:
df_check['platform'] = df_check['platform'].str.strip().str.lower()
mapping = {'iphone': 'ios'}
df_check['platform'] = df_check['platform'].replace(mapping)

## Посмотрим на дубликаты

In [ ]:
df_check.duplicated().sum()

np.int64(2954)

In [ ]:
df_check.duplicated(
    ['cookie_id', 'event_ts', 'eid', 'event_name']
).sum()

np.int64(3202)

Сравним долю дублей по таргетам

In [ ]:
df_check[df_check['target'] == 1].duplicated().sum()/df_check[df_check['target'] == 1].shape[0]

np.float64(0.015165805258988191)

In [ ]:
df_check[df_check['target'] == 0].duplicated().sum()/df_check[df_check['target'] == 0].shape[0]

np.float64(0.01484333649355257)

Были обнаружены полные дубликаты строк событий. Они не несут дополнительной информации и могут искусственно увеличивать частоты событий, поэтому перед построением агрегатных признаков они удаляются.

In [ ]:
df_check = df_check.drop_duplicates()

In [ ]:
df_check[df_check['target'] == 1].duplicated(
    ['cookie_id', 'event_ts', 'eid', 'event_name']
).sum()/df_check[df_check['target'] == 1].shape[0]

np.float64(0.00030645470216433634)

In [ ]:
df_check[df_check['target'] == 0].duplicated(
    ['cookie_id', 'event_ts', 'eid', 'event_name']
).sum()/df_check[df_check['target'] == 0].shape[0]

np.float64(0.0014169574381409518)

In [ ]:
mask = (df_check['target'] == 1) & \
       df_check.duplicated(['cookie_id', 'event_ts', 'eid', 'event_name'], keep=False)

df_check[mask].sort_values(by='event_ts')

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts,target
136168,ck_4af824883d103e60,2026-04-10 14:01:32,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1039629.0,uslugi,novosibirsk,pro,NaN,NaN,395.0,510.0,2026-04-10,2026-04-11,1
138677,ck_4af824883d103e60,2026-04-10 14:01:32,200,item_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1028989.0,uslugi,penza,pro,NaN,NaN,782.0,126.0,2026-04-10,2026-04-11,1
15385,ck_ec208e9e80567277,2026-04-11 08:46:28,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,bytovaya_tehnika,moskva,NaN,микроволновка,4.0,806.0,560.0,2026-04-11,2026-04-12,1
51629,ck_ec208e9e80567277,2026-04-11 08:46:28,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,NaN,perm,NaN,студия купить,6.0,692.0,603.0,2026-04-11,2026-04-12,1
30685,ck_37d99ec8005d1a28,2026-04-13 11:34:47,210,photo_swipe,ios,Avito/118.0 (iPhone; iOS 17.2; Scale/3.00),1058842.0,bytovaya_tehnika,moskva,pro,NaN,NaN,NaN,NaN,2026-04-13,2026-04-14,1
125927,ck_37d99ec8005d1a28,2026-04-13 11:34:47,210,photo_swipe,ios,Avito/118.0 (iPhone; iOS 17.2; Scale/3.00),1001526.0,bytovaya_tehnika,saratov,private,NaN,NaN,NaN,NaN,2026-04-13,2026-04-14,1
97027,ck_afe940c75ca49ea3,2026-04-13 15:25:27,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,NaN,makhachkala,NaN,шкаф купе,6.0,NaN,NaN,2026-04-13,2026-04-14,1
186854,ck_afe940c75ca49ea3,2026-04-13 15:25:27,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,mebel,novosibirsk,NaN,стол письменный,1.0,NaN,NaN,2026-04-13,2026-04-14,1
131710,ck_6e9308935da54123,2026-04-15 17:20:04,200,item_view,desktop,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1008399.0,noutbuki,ekaterinburg,pro,NaN,NaN,NaN,NaN,2026-04-15,2026-04-16,1
172318,ck_6e9308935da54123,2026-04-15 17:20:04,200,item_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1058599.0,mebel,ekaterinburg,pro,NaN,NaN,NaN,NaN,2026-04-15,2026-04-16,1


In [ ]:
mask = (df_check['target'] == 0) & \
       df_check.duplicated(['cookie_id', 'event_ts', 'eid', 'event_name'], keep=False)

df_check[mask].sort_values(by='event_ts')

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y,window_start_ts,window_end_ts,target
129141,ck_76bfa35aad18f7ec,2026-04-06 05:24:43,210,photo_swipe,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,1015243.0,mebel,voronezh,private,NaN,NaN,390.0,241.0,2026-04-06,2026-04-07,0
52398,ck_76bfa35aad18f7ec,2026-04-06 05:24:43,210,photo_swipe,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,1038078.0,mebel,bryansk,pro,NaN,NaN,1204.0,82.0,2026-04-06,2026-04-07,0
69962,ck_f34ad37fb9c3df05,2026-04-06 06:29:30,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,kvartiry_arenda,vladivostok,NaN,квартира посуточно,1.0,1845.0,1047.0,2026-04-06,2026-04-07,0
62076,ck_f34ad37fb9c3df05,2026-04-06 06:29:30,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,kvartiry_arenda,ekaterinburg,NaN,снять квартиру у метро,1.0,NaN,NaN,2026-04-06,2026-04-07,0
162690,ck_c8a1ee8c28436857,2026-04-06 08:55:25,100,search_results_view,IOS,Avito/118.0 (iPhone; iOS 17.2; Scale/3.00),NaN,hobbi,samara,NaN,гитара акустическая,1.0,NaN,NaN,2026-04-06,2026-04-07,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144914,ck_213fcc239f67793e,2026-04-19 20:30:58,200,item_view,web,Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/53...,1005336.0,odezhda,ekaterinburg,private,NaN,NaN,1735.0,147.0,2026-04-19,2026-04-20,0
150366,ck_210e8021b93a0c1d,2026-04-19 23:39:27,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,kvartiry_arenda,novosibirsk,NaN,снять 1к квартиру,1.0,1022.0,306.0,2026-04-19,2026-04-20,0
78056,ck_210e8021b93a0c1d,2026-04-19 23:39:27,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,NaN,kvartiry_arenda,novosibirsk,NaN,снять квартиру у метро,3.0,1046.0,97.0,2026-04-19,2026-04-20,0
103740,ck_f5cc52e7b83bacde,2026-04-19 23:50:29,100,search_results_view,ANDROID,Mozilla/5.0 (Linux; Android 11; SM-G991B) Appl...,NaN,NaN,kaliningrad,NaN,стол письменный,2.0,NaN,NaN,2026-04-19,2026-04-20,0


## Общая активность

Количество уникальных типов событий

In [ ]:
df_check[df_check['target'] == 1].groupby('cookie_id')['event_name'].nunique().describe()

,event_name
count,899.000000
mean,4.803115
std,1.634862
min,1.000000
25%,4.000000
50%,5.000000
75%,6.000000
max,9.000000


In [ ]:
px.histogram(df_check[df_check['target'] == 1].groupby('cookie_id')['event_name'].nunique())

In [ ]:
df_check[df_check['target'] == 0].groupby('cookie_id')['event_name'].nunique().describe()

,event_name
count,10192.000000
mean,4.598312
std,1.915840
min,1.000000
25%,3.000000
50%,4.000000
75%,6.000000
max,9.000000


In [ ]:
px.histogram(df_check[df_check['target'] == 0].groupby('cookie_id')['event_name'].nunique())

У обычных пользователей разнообразие действий в среднем меньше, чем у ботов (у ботов получается в целом нормальное распределение, у обычных пользователей скос больше влево все-таки), а также разброс по количеству действий у пользователей больше, чем у ботов



---

Сравним количество событий по группам

In [ ]:
df_check[df_check['target'] == 1].groupby('cookie_id')['event_name'].count().describe()

,event_name
count,899.000000
mean,29.037820
std,27.082212
min,1.000000
25%,11.000000
50%,20.000000
75%,37.000000
max,165.000000


In [ ]:
px.histogram(df_check[df_check['target'] == 1].groupby('cookie_id')['event_name'].count())

In [ ]:
df_check[df_check['target'] == 0].groupby('cookie_id')['event_name'].count().describe()

,event_name
count,10192.000000
mean,16.618622
std,15.929946
min,1.000000
25%,6.000000
50%,12.000000
75%,22.000000
max,137.000000


In [ ]:
px.histogram(df_check[df_check['target'] == 0].groupby('cookie_id')['event_name'].count())

В среднем у ботов действий больше, чем у пользователей.

---


Посчитаем уникальные часы активности для каждой группы

In [ ]:
df_check['event_hour'] = df_check['event_ts'].dt.hour

active_hours_target_1 = df_check[df_check['target'] == 1].groupby('cookie_id')['event_hour'].nunique()
print('Активные часы для ботов:')
print(active_hours_target_1.describe())
px.histogram(active_hours_target_1).show()

active_hours_target_0 = df_check[df_check['target'] == 0].groupby('cookie_id')['event_hour'].nunique()
print('\nАктивные часы для пользователей:')
print(active_hours_target_0.describe())
px.histogram(active_hours_target_0).show()

Активные часы для ботов:
count    899.000000
mean       3.221357
std        2.032298
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max       16.000000
Name: event_hour, dtype: float64



Активные часы для пользователей:
count    10192.000000
mean         3.132457
std          2.237870
min          1.000000
25%          2.000000
50%          3.000000
75%          4.000000
max         17.000000
Name: event_hour, dtype: float64


В целом и у пользователей и у ботов не такое большое разнообразие уникальных часов активности



---
Посмотрим, когда активнее каждая группа в течение суток


In [ ]:
df_hour = df_check.copy()

df_hour['event_ts'] = pd.to_datetime(df_hour['event_ts'])
df_hour['hour'] = df_hour['event_ts'].dt.hour

hour_counts = (
    df_hour
    .groupby(['cookie_id', 'hour'])
    .size()
    .unstack(fill_value=0)
)

hour_counts = hour_counts.reindex(columns=range(24), fill_value=0)

hour_shares = hour_counts.div(
    hour_counts.sum(axis=1),
    axis=0
)

hour_shares = hour_shares.reset_index()

hour_shares = hour_shares.merge(
    df_hour[['cookie_id', 'target']].drop_duplicates('cookie_id'),
    on='cookie_id',
    how='left'
)

hour_shares.head()

,cookie_id,0,1,2,3,4,5,6,7,8,...,15,16,17,18,19,20,21,22,23,target
0,ck_000c95f1408dcb00,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.187500,0.125000,0.000000,0.000000,0.437500,0.000,0.0625,0.0,0.0625,0
1,ck_000e8c52636e3bec,0.0,0.0,0.000000,0.000000,0.0,0.0,0.117647,0.441176,0.441176,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000,0.0000,0.0,0.0000,0
2,ck_0010e31baa4a1fb7,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000,0.062500,0.062500,0.062500,0.000000,0.125,0.0625,0.0,0.2500,0
3,ck_0010ec3874fb5378,0.0,0.0,0.027027,0.189189,0.0,0.0,0.013514,0.027027,0.000000,...,0.067568,0.067568,0.175676,0.000000,0.000000,0.000,0.0000,0.0,0.0000,0
4,ck_001722063b94cae0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.666667,0.333333,0.000,0.0000,0.0,0.0000,0


In [ ]:
import matplotlib.pyplot as plt

In [ ]:
hour_mean = (
    hour_shares
    .groupby('target')[list(range(24))]
    .mean()
    .T
)

hour_mean.columns = ['target_0', 'target_1']


In [ ]:
plot_df = (
    hour_mean
    .reset_index(names='hour')
    .melt(id_vars='hour',
          value_vars=['target_0', 'target_1'],
          var_name='target', value_name='share')
)


In [ ]:
px.line(
    plot_df, x='hour', y='share', color='target', markers=True,
    title='Профиль активности по часам',
).update_xaxes(dtick=1).show()

In [ ]:
hour_mean['difference'] = (
    hour_mean['target_1'] - hour_mean['target_0']
)

hour_mean[['target_0', 'target_1', 'difference']]

,target_0,target_1,difference
0,0.012713,0.020111,0.007398
1,0.017664,0.034757,0.017093
2,0.021759,0.040099,0.018340
3,0.025755,0.049978,0.024223
4,0.026879,0.038011,0.011132
5,0.030211,0.039361,0.009149
6,0.026294,0.029575,0.003281
7,0.029709,0.031768,0.002059
8,0.031995,0.032961,0.000966
9,0.036253,0.033109,-0.003143


Пользователи более активны во второй половине дня, а активность ботов сначала выше, а затем начинает расти с пользовательской активностью

Попробуем укрупнить результаты

In [ ]:
def get_time_period(hour):
    if 0 <= hour < 6:
        return 'night'
    elif 6 <= hour < 12:
        return 'morning'
    elif 12 <= hour < 18:
        return 'afternoon'
    else:
        return 'evening'


df_hour['time_period'] = df_hour['hour'].apply(get_time_period)

period_counts = (
    df_hour
    .groupby(['cookie_id', 'time_period'])
    .size()
    .unstack(fill_value=0)
)

period_counts = period_counts.reindex(
    columns=['night', 'morning', 'afternoon', 'evening'],
    fill_value=0
)

period_shares = period_counts.div(
    period_counts.sum(axis=1),
    axis=0
)

period_shares = period_shares.reset_index()

period_shares = period_shares.merge(
    df_hour[['cookie_id', 'target']].drop_duplicates('cookie_id'),
    on='cookie_id',
    how='left'
)

period_summary = (
    period_shares
    .groupby('target')[['night', 'morning', 'afternoon', 'evening']]
    .mean()
    .T
)

period_summary.columns = ['target_0', 'target_1']

period_summary['difference'] = (
    period_summary['target_1'] - period_summary['target_0']
)

period_summary

,target_0,target_1,difference
night,0.134980,0.222315,0.087335
morning,0.205546,0.197148,-0.008398
afternoon,0.276202,0.236368,-0.039834
evening,0.383272,0.344168,-0.039104


В целом и по укрупненным группам заметна разница в поведении, признаки с difference вполне можно использовать

## События

Посчитаем сколько каждого видов событий происходило по группам

In [ ]:
event_counts = pd.crosstab(
    df_check['cookie_id'],
    df_check['event_name']
)

# Добавляем target
event_counts = event_counts.merge(
    df_check[['cookie_id', 'target']].drop_duplicates('cookie_id'),
    left_index=True,
    right_on='cookie_id',
    how='left'
)

event_counts.head()

,contact_chat_open,contact_message_sent,contact_phone_show,favorite_add,item_view,login,photo_swipe,search_results_view,seller_page_view,cookie_id,target
157218,0,0,0,0,11,0,1,3,1,ck_000c95f1408dcb00,0
114556,0,0,1,4,11,1,5,9,3,ck_000e8c52636e3bec,0
48361,0,0,0,2,7,1,1,4,1,ck_0010e31baa4a1fb7,0
95950,0,0,0,10,25,1,14,20,4,ck_0010ec3874fb5378,0
167841,0,0,1,0,8,0,1,5,0,ck_001722063b94cae0,0


In [ ]:
event_count_summary = (
    event_counts
    .groupby('target')
    .mean(numeric_only=True)
    .T
    .sort_values(1, ascending=False)
)

event_count_summary

target,0,1
item_view,6.062500,12.676307
search_results_view,5.065640,10.208009
photo_swipe,2.039246,2.028921
seller_page_view,0.883634,1.717464
favorite_add,1.092131,0.791991
contact_phone_show,0.617249,0.745273
contact_chat_open,0.329965,0.414905
login,0.357339,0.228031
contact_message_sent,0.170918,0.226919


In [ ]:
px.bar(event_count_summary, barmode='overlay')

В среднем item_view и search_results_view действий в 2 раза больше совершают боты

---

Сравним доли каждого события относительно всех событий по группам

In [ ]:
event_cols = [
    col for col in event_counts.columns
    if col not in ['cookie_id', 'target']
]

event_shares = event_counts.copy()

event_shares[event_cols] = (
    event_shares[event_cols]
    .div(event_shares[event_cols].sum(axis=1), axis=0)
)

event_share_summary = (
    event_shares
    .groupby('target')[event_cols]
    .mean()
    .T
    .sort_values(1, ascending=False)
)

event_share_summary

target,0,1
item_view,0.364103,0.424915
search_results_view,0.308335,0.331245
photo_swipe,0.127098,0.078010
seller_page_view,0.052948,0.061275
contact_phone_show,0.030824,0.033203
favorite_add,0.068684,0.033059
contact_chat_open,0.016733,0.018185
contact_message_sent,0.008086,0.010203
login,0.023188,0.009905


Доля item_view для ботов выше, чем для пользователей. А photo_swipe, favorite_add и login больше в действиях обычных людей.



---

Сравним относительные частоты событий

In [ ]:
per_cookie = (df_check[df_check['target'] == 1]
              .groupby(['cookie_id', 'event_name']).size()
              .unstack(fill_value=0))

# сглаживаем нули, чтобы не делить на 0 и не получать inf
eps = 1e-9
ratio_per_cookie = (per_cookie.values[:, :, None] + eps)/ (per_cookie.values[:, None, :] + eps)


median_ratio = np.median(ratio_per_cookie, axis=0)

ratio_df = pd.DataFrame(median_ratio,
                        index=per_cookie.columns,
                        columns=per_cookie.columns)

In [ ]:
log_ratio = np.log2(ratio_df)

In [ ]:
fig = px.imshow(log_ratio)
fig.show()

In [ ]:
per_cookie = (df_check[df_check['target'] == 0]
              .groupby(['cookie_id', 'event_name']).size()
              .unstack(fill_value=0))

# сглаживаем нули, чтобы не делить на 0 и не получать inf
eps = 1e-9
ratio_per_cookie = (per_cookie.values[:, :, None] + eps)/ (per_cookie.values[:, None, :] + eps)


median_ratio = np.median(ratio_per_cookie, axis=0)

ratio_df = pd.DataFrame(median_ratio,
                        index=per_cookie.columns,
                        columns=per_cookie.columns)
log_ratio = np.log2(ratio_df)

In [ ]:
fig = px.imshow(log_ratio)
fig.show()

Получается для ботов отношение search_result_view к favorite_add, seller_page_view к contact_message_sent, item_view к favorite_add, seller_page_view к login выше, чем у обычных пользователей (обычные пользователи чаще добавляют в избранное относительно просмотра товара, чаще пишут продавцу относительно просмотра его профиля и тд)



---



## Объявления

Сравним количество просмотров уникальных объявлений по группам

In [ ]:
df_check[df_check['target']==1].groupby('cookie_id')['item_id'].nunique().describe()

,item_id
count,899.000000
mean,13.779755
std,10.983506
min,1.000000
25%,6.000000
50%,10.000000
75%,18.000000
max,70.000000


In [ ]:
px.histogram(df_check[df_check['target']==1].groupby('cookie_id')['item_id'].nunique())

In [ ]:
df_check[df_check['target']==0].groupby('cookie_id')['item_id'].nunique().describe()

,item_id
count,10192.000000
mean,8.917190
std,9.018032
min,0.000000
25%,3.000000
50%,6.000000
75%,12.000000
max,80.000000


In [ ]:
px.histogram(df_check[df_check['target']==0].groupby('cookie_id')['item_id'].nunique())

В среднем у ботов просмотренных уникальных объявлений больше, чем у обычных пользователей



---

Сравним количество действий с каждым объявлением

In [ ]:
df_check[df_check['target']==1].groupby(['cookie_id', 'item_id'])['event_name'].count().describe()

,event_name
count,12388.000000
mean,1.349935
std,0.670211
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,6.000000


In [ ]:
px.histogram(df_check[df_check['target']==1].groupby(['cookie_id', 'item_id'])['event_name'].count())

In [ ]:
df_check[df_check['target']==0].groupby(['cookie_id', 'item_id'])['event_name'].count().describe()

,event_name
count,90884.000000
mean,1.255513
std,0.545625
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,7.000000


In [ ]:
px.histogram(df_check[df_check['target']==0].groupby(['cookie_id', 'item_id'])['event_name'].count())



---

Сравним разнообразие действий с каждым объявлением

In [ ]:
df_check[df_check['target']==1].groupby(['cookie_id', 'item_id'])['event_name'].nunique().describe()

,event_name
count,12388.000000
mean,1.142557
std,0.377823
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,4.000000


In [ ]:
px.histogram(df_check[df_check['target']==1].groupby(['cookie_id', 'item_id'])['event_name'].nunique())

In [ ]:
df_check[df_check['target']==0].groupby(['cookie_id', 'item_id'])['event_name'].nunique().describe()

,event_name
count,90884.000000
mean,1.154780
std,0.395469
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,4.000000


In [ ]:
px.histogram(df_check[df_check['target']==0].groupby(['cookie_id', 'item_id'])['event_name'].nunique())

В текущем агрегированном сравнении выраженного различия не видно. В дальнейшем можно проверить полезность в модели.

## География



---

Посмотрим, насколько локализованно смотрят люди и боты

In [ ]:
# Но сначала проверим согласованность данных
df_check['item_location'].unique()

array(['kirov', 'novosibirsk', 'sankt-peterburg', 'ekaterinburg',
       'bryansk', 'samara', 'tolyatti', 'habarovsk', 'volgograd',
       'krasnodar', 'omsk', 'kemerovo', 'irkutsk', 'tula', 'penza', nan,
       'tyumen', 'moskva', 'saratov', 'yaroslavl', 'ufa', 'krasnoyarsk',
       'nizhniy_novgorod', 'barnaul', 'chelyabinsk', 'izhevsk', 'kazan',
       'vladivostok', 'rostov-na-donu', 'kaliningrad', 'ivanovo',
       'makhachkala', 'ryazan', 'perm', 'ulyanovsk', 'lipetsk',
       'cheboksary', 'astrahan', 'voronezh', 'orenburg', 'tomsk'],
      dtype=object)

In [ ]:
df_check[df_check['target']==1].groupby('cookie_id')['item_location'].nunique().describe()

,item_location
count,899.000000
mean,9.789766
std,6.068215
min,1.000000
25%,5.000000
50%,8.000000
75%,14.000000
max,31.000000


In [ ]:
px.histogram(df_check[df_check['target']==1].groupby('cookie_id')['item_location'].nunique())

In [ ]:
df_check[df_check['target']==0].groupby('cookie_id')['item_location'].nunique().describe()

,item_location
count,10192.000000
mean,5.881868
std,4.637013
min,0.000000
25%,2.000000
50%,5.000000
75%,8.000000
max,30.000000


In [ ]:
px.histogram(df_check[df_check['target']==0].groupby('cookie_id')['item_location'].nunique())

Среднее значение просмотренных городов для бота сильно больше (9.79), чем у обычного пользователя (5.88), то же самое и с разбросом. Само рапределение для ботов имеет тяжелый хвост справа.



---



Посмотрим на соотношение самого часто встречаемого города к общему количеству

In [ ]:

counts = df_check[df_check['target'] == 1].groupby('cookie_id')['item_location'].value_counts()

shares = counts / counts.groupby('cookie_id').transform('sum')

top_city = (
    shares
    .reset_index(name='share')
    .sort_values(['cookie_id', 'share'], ascending=[True, False])
    .groupby('cookie_id', as_index=False)
    .head(1)
)

top_city.describe()

,share
count,899.000000
mean,0.379468
std,0.212406
min,0.091549
25%,0.200000
50%,0.333333
75%,0.500000
max,1.000000


In [ ]:
px.histogram(top_city['share'])

In [ ]:

counts = df_check[df_check['target'] == 0].groupby('cookie_id')['item_location'].value_counts()

shares = counts / counts.groupby('cookie_id').transform('sum')

top_city = (
    shares
    .reset_index(name='share')
    .sort_values(['cookie_id', 'share'], ascending=[True, False])
    .groupby('cookie_id', as_index=False)
    .head(1)
)

top_city.describe()

,share
count,10177.000000
mean,0.536102
std,0.253242
min,0.100000
25%,0.318182
50%,0.500000
75%,0.733333
max,1.000000


In [ ]:
px.histogram(top_city['share'])

У ботов куда больше разброса по городам, реже ищут в каком-то конкретном городе, тогда как у пользователей много и тех, кто ищет только в одном городе

## Категории

Посмотрим на количество уникальных категорий, просматриваемых каждой из групп

In [ ]:
df_check[df_check['target']==1].groupby('cookie_id')['item_category'].nunique().describe()

,item_category
count,899.000000
mean,1.787542
std,0.814963
min,0.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,6.000000


In [ ]:
px.histogram(df_check[df_check['target']==1].groupby('cookie_id')['item_category'].nunique())

In [ ]:
df_check[df_check['target']==0].groupby('cookie_id')['item_category'].nunique().describe()

,item_category
count,10192.000000
mean,2.281201
std,1.257226
min,0.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,8.000000


In [ ]:
px.histogram(df_check[df_check['target']==0].groupby('cookie_id')['item_category'].nunique())

В среднем различных категорий у пользователей больше, чем у ботов, разброс у пользователей так же больше

## Поиск

Сравним количество поисковых запросов у разных групп пользователей

In [ ]:
df_check[df_check['target']==1].groupby('cookie_id')['search_query'].nunique().describe()

,search_query
count,899.000000
mean,4.758621
std,3.142606
min,0.000000
25%,2.000000
50%,4.000000
75%,7.000000
max,17.000000


In [ ]:
px.histogram(df_check[df_check['target']==1].groupby('cookie_id')['search_query'].nunique())

In [ ]:
df_check[df_check['target']==0].groupby('cookie_id')['search_query'].nunique().describe()

,search_query
count,10192.000000
mean,3.789835
std,3.200398
min,0.000000
25%,1.000000
50%,3.000000
75%,5.000000
max,29.000000


In [ ]:
px.histogram(df_check[df_check['target']==0].groupby('cookie_id')['search_query'].nunique())

В среднем у пользователей меньше поисковых запросов

---
Сравним количество запросов в каждой категории

In [ ]:
df_check[df_check['target']==1].groupby(['cookie_id', 'item_category'])['search_query'].nunique().describe()

,search_query
count,1607.000000
mean,2.558805
std,2.167086
min,0.000000
25%,1.000000
50%,2.000000
75%,4.000000
max,8.000000


In [ ]:
px.histogram(df_check[df_check['target']==1].groupby(['cookie_id', 'item_category'])['search_query'].nunique())

In [ ]:
df_check[df_check['target']==0].groupby(['cookie_id', 'item_category'])['search_query'].nunique().describe()

,search_query
count,23250.000000
mean,1.581075
std,1.513437
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,8.000000


In [ ]:
px.histogram(df_check[df_check['target']==0].groupby(['cookie_id', 'item_category'])['search_query'].nunique())

У ботов разнообразие запросов относительно категорий выше, признак можно в дальнейшем проверить на модели

---
Сравним длину поискового запроса

In [ ]:
df_check[df_check['target']==1].groupby('cookie_id')['search_query'].apply(lambda s: s.str.split().str.len().mean()).describe()

,search_query
count,862.000000
mean,1.995028
std,0.448106
min,1.000000
25%,1.750000
50%,2.000000
75%,2.181006
max,4.000000


In [ ]:
px.histogram(df_check[df_check['target']==1].groupby('cookie_id')['search_query'].apply(lambda s: s.str.split().str.len().mean()))

In [ ]:
df_check[df_check['target']==0].groupby('cookie_id')['search_query'].apply(lambda s: s.str.split().str.len().mean()).describe()

,search_query
count,9308.000000
mean,1.988808
std,0.448805
min,1.000000
25%,1.750000
50%,2.000000
75%,2.166667
max,4.000000


In [ ]:
px.histogram(df_check[df_check['target']==0].groupby('cookie_id')['search_query'].apply(lambda s: s.str.split().str.len().mean()))

В среднем и для ботов, и для пользователей длина запроса одинакова (~2 слова), полезность можно проверить на модели в дальнейшем



---

Рассмотрим среднюю страницу, на которой каждая группа открывает объявление

In [ ]:
df_check[df_check['target'] == 1].groupby(['cookie_id'])['search_page'].mean().describe()

,search_page
count,862.000000
mean,3.587105
std,1.870407
min,1.000000
25%,2.135714
50%,3.190909
75%,4.641234
max,12.333333


In [ ]:
px.histogram(df_check[df_check['target'] == 1].groupby(['cookie_id'])['search_page'].mean())

In [ ]:
df_check[df_check['target'] == 0].groupby(['cookie_id'])['search_page'].mean().describe()

,search_page
count,9308.000000
mean,2.315803
std,1.119573
min,1.000000
25%,1.500000
50%,2.000000
75%,3.000000
max,13.000000


In [ ]:
px.histogram(df_check[df_check['target'] == 0].groupby(['cookie_id'])['search_page'].mean())

Обычные пользователи все-таки чаще всего нажимают на объявление на первой странице. Ну и среднее значение для пользователей меньше, чем для ботов, и распределение не такое однородное. Мера разброса также меньше для пользователей.



---



Сравним сколько запросов в среднем повторялось

In [ ]:
bots = df_check[df_check['target'] == 1]

per_cookie = (
    bots.groupby('cookie_id')['search_query']
          .apply(lambda s: s.duplicated().mean())
)

per_cookie.describe()

,search_query
count,899.000000
mean,0.713548
std,0.164871
min,0.000000
25%,0.642857
50%,0.750000
75%,0.828079
max,0.938356


In [ ]:
px.histogram(per_cookie)

In [ ]:
humans = df_check[df_check['target'] == 0]

per_cookie = (
    humans.groupby('cookie_id')['search_query']
          .apply(lambda s: s.duplicated().mean())
)

per_cookie.describe()

,search_query
count,10192.000000
mean,0.608049
std,0.211798
min,0.000000
25%,0.500000
50%,0.666667
75%,0.750000
max,0.941176


In [ ]:
px.histogram(per_cookie)

В среднем боты чаще повторяют запросы, чем обычные пользователи, их распределение более однородное, стремится к большим значениям

## Характеристики для времени

Посчитаем среднее время между любыми событиями по группам

In [ ]:
df_check = df_check.sort_values(['cookie_id', 'event_ts'])

In [ ]:
sub_bots = df_check[(df_check['target'] == 1)].copy()

sub_bots['delta'] = sub_bots.groupby('cookie_id')['event_ts'].diff()

avg = (
    sub_bots.groupby('cookie_id')['delta']
       .mean()
       .dt.total_seconds()
       .rename('avg_seconds')
       .reset_index()
)
avg.describe()

,avg_seconds
count,894.000000
mean,571.273388
std,543.630483
min,4.250000
25%,192.292763
50%,429.714286
75%,802.736364
max,3777.500000


In [ ]:
px.histogram(avg['avg_seconds'])

In [ ]:
sub = df_check[(df_check['target'] == 0)].copy()

sub['delta'] = sub.groupby('cookie_id')['event_ts'].diff()

avg = (
    sub.groupby('cookie_id')['delta']
       .mean()
       .dt.total_seconds()
       .rename('avg_seconds')
       .reset_index()
)
avg.describe()

,avg_seconds
count,9943.000000
mean,852.444252
std,876.717217
min,0.000000
25%,228.832418
50%,670.692308
75%,1150.083333
max,9005.000000


In [ ]:
px.histogram(avg['avg_seconds'])

Среднее время между всеми действиями для пользователя в больше, для ботов значение имеет экспоненциальное распределение



---

Сравним энтропию для поведения каждой группы

In [ ]:
sub = df_check[df_check['target'] == 1].sort_values(['cookie_id', 'event_ts']).copy()

sub['delta'] = sub.groupby('cookie_id')['event_ts'].diff()

g = sub.groupby('cookie_id')['delta']


In [ ]:
def interval_entropy(x, bins=20):
    sec = x.dt.total_seconds().dropna()
    if len(sec) < 2:
        return np.nan
    hist, _ = np.histogram(sec, bins=bins)
    p = hist / hist.sum()
    p = p[p > 0]
    return -(p * np.log2(p)).sum()

delta_entropy = g.apply(interval_entropy)
delta_entropy.describe()

,delta
count,885.000000
mean,1.086037
std,0.895068
min,-0.000000
25%,0.439497
50%,0.755418
75%,1.469582
max,3.732445


In [ ]:
px.histogram(delta_entropy)

In [ ]:
sub = df_check[df_check['target'] == 0].sort_values(['cookie_id', 'event_ts']).copy()

sub['delta'] = sub.groupby('cookie_id')['event_ts'].diff()

g = sub.groupby('cookie_id')['delta']

delta_entropy = g.apply(interval_entropy)
delta_entropy.describe()

,delta
count,9556.000000
mean,1.200841
std,0.664502
min,-0.000000
25%,0.721928
50%,1.000000
75%,1.521641
max,3.984295


In [ ]:
px.histogram(delta_entropy)

В целом распределение для ботов ближе к нулю, чем для пользователей, то есть поведение ботов более предсказуемое

## Активность по времени

Максимальные значения по коротким промежуткам времени

In [ ]:
def max_events_per_window(df, windows=('1min', '5min', '30min')):
    df = df.sort_values(['cookie_id', 'event_ts']).copy()
    out = None

    for w in windows:
        col = f'max_events_{w}'
        s = (
            df.set_index('event_ts')
              .groupby('cookie_id')['event_name']
              .rolling(w)
              .count()
              .reset_index(level=0, drop=True)
        )
        per_cookie = s.groupby(df['cookie_id'].values).max().rename(col)
        out = per_cookie.to_frame() if out is None else out.join(per_cookie)

    return out

feat_bots = max_events_per_window(df_check[df_check['target'] == 1])
feat_bots.describe()

,max_events_1min,max_events_5min,max_events_30min
count,899.000000,899.000000,899.00000
mean,4.844271,12.293660,15.85317
std,3.207764,10.701689,13.85844
min,1.000000,1.000000,1.00000
25%,3.000000,5.000000,6.00000
50%,4.000000,10.000000,12.00000
75%,6.000000,16.000000,20.00000
max,24.000000,84.000000,129.00000


In [ ]:
px.histogram(feat_bots)

In [ ]:
feat_human = max_events_per_window(df_check[df_check['target'] == 0])
feat_human.describe()

,max_events_1min,max_events_5min,max_events_30min
count,10192.000000,10192.000000,10192.000000
mean,2.925432,5.822508,8.735086
std,1.271632,3.279836,6.317265
min,1.000000,1.000000,1.000000
25%,2.000000,3.000000,4.000000
50%,3.000000,5.000000,7.000000
75%,4.000000,7.000000,11.000000
max,11.000000,27.000000,57.000000


In [ ]:
px.histogram(feat_human)

У ботов, ожидаемо, в среднем больше действий за котороткий промежуток времени, больше разброса, есть очень большие значения (130 действий за 30 минут)

## User_agent

Посмотрим, можно ли что-то взять из этого признака

Это фича, которая может содержать информацию о ОС, браузере версии браузера, устройстве, движке, мобильном приложении.

In [ ]:
df_check[df_check['target'] == 1]['user_agent'].value_counts().head(5)

,count
user_agent,
"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/118.0.0.0 Safari/537.36",1206
"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/114.0.0.0 Safari/537.36",1016
"Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) HeadlessChrome/120.0.0.0 Safari/537.36",786
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",670
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36",582


In [ ]:
df_check[df_check['target'] == 0]['user_agent'].value_counts().head(5)

,count
user_agent,
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36",2112
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/121.0.0.0 YaBrowser/23.1.0.0 Safari/537.36",1878
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",1868
"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",1854
"Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",1848


В самых популярных значениях у ботов часто встречается HeadlessChrome, посмотрим, есть ли взаимосвязь

In [ ]:
mask = df_check['user_agent'].str.contains('HeadlessChrome', case=False, na=False)

result = mask.groupby(df_check['target']).sum()/mask.groupby(df_check['target']).count()
result

,user_agent
target,
0,0.020381
1,0.132197


В целом доля значений с HeadlessChrome у ботов выше, чем у обычных пользователей

Посмотрим, есть ли еще какие-то особенности у каждой группой

In [ ]:
ua = df_check['user_agent'].fillna('').str.lower()

features = pd.DataFrame({
    'ua_has_headless': ua.str.contains('headlesschrome', regex=False),
    'ua_is_yabrowser': ua.str.contains('yabrowser',    regex=False),
    'ua_is_chrome':    ua.str.contains('chrome',       regex=False) & ~ua.str.contains('headless', regex=False),
    'ua_is_firefox':   ua.str.contains('firefox',      regex=False),
    'ua_is_safari':    ua.str.contains('safari',       regex=False) & ~ua.str.contains('chrome', regex=False),
    'ua_is_avito_app': ua.str.contains('avito',        regex=False),
    'ua_is_android':   ua.str.contains('android',      regex=False),
    'ua_is_ios':       ua.str.contains('iphone|ipad|ipod', regex=True),
    'ua_is_windows':   ua.str.contains('windows',      regex=False),
    'ua_is_mac':       ua.str.contains('macintosh|mac os x', regex=True),
    'ua_is_linux':     ua.str.contains('linux',        regex=False) & ~ua.str.contains('android', regex=False),
}, index=df_check.index)

share = features.groupby(df_check['target']).mean().T
share.columns = ['share_humans', 'share_bots']
share['diff'] = share['share_bots'] - share['share_humans']
share['ratio'] = share['share_bots'] / share['share_humans'].replace(0, pd.NA)

print(share.sort_values('diff', ascending=False).round(4))

                 share_humans  share_bots    diff   ratio
ua_has_headless        0.0204      0.1322  0.1118  6.4864
ua_is_linux            0.0204      0.1322  0.1118  6.4864
ua_is_windows          0.3631      0.4291  0.0660  1.1818
ua_is_firefox          0.1239      0.1571  0.0332  1.2682
ua_is_yabrowser        0.1149      0.1255  0.0106  1.0919
ua_is_safari           0.0189      0.0091 -0.0098  0.4805
ua_is_mac              0.1462      0.1254 -0.0208  0.8577
ua_is_avito_app        0.1888      0.1664 -0.0224  0.8815
ua_is_ios              0.0575      0.0208 -0.0367  0.3620
ua_is_chrome           0.6337      0.4683 -0.1654  0.7389
ua_is_android          0.4174      0.2347 -0.1827  0.5622


Строки, содержащие HeadlessChrome, Linux, чаще относятся к ботам, а содержащие android, ios, safari, - к обычным пользователям



---
Сравним количество уникальных значений у разных групп


In [ ]:
df_check[df_check['target'] == 1].groupby('cookie_id')['user_agent'].nunique().describe()

,user_agent
count,899.000000
mean,1.112347
std,0.459559
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,3.000000


In [ ]:
df_check[df_check['target'] == 0].groupby('cookie_id')['user_agent'].nunique().describe()

,user_agent
count,10192.000000
mean,1.135793
std,0.492716
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,3.000000


В целом заметных различий для данного набора данных здесь нет

## Seller_type

Сравним, есть ли взаимосвязь между типом продавца и таргетом

In [ ]:
px.bar(df_check[df_check['target'] == 1].groupby(['cookie_id', 'seller_type']).size().groupby('seller_type').mean())

In [ ]:
px.bar(df_check[df_check['target'] == 0].groupby(['cookie_id', 'seller_type']).size().groupby('seller_type').mean())

Заметных различий нет, соотношение одинаковое (если не уделять внимание тому, что у ботов в целом больше действий и просмотров продавцов больше), но можно оставить признак до прогона модели

## Pointer

Рассмотрим, что нам могут рассказать координаты курсора в момент события



---

Посчитаем расстояния кликов, уникальных кликов

In [ ]:
df_pointer = df_check.copy()

df_pointer['event_ts'] = pd.to_datetime(df_pointer['event_ts'])

df_pointer = df_pointer.sort_values(
    ['cookie_id', 'event_ts']
)

df_pointer['prev_x'] = (
    df_pointer
    .groupby('cookie_id')['pointer_x']
    .shift(1)
)

df_pointer['prev_y'] = (
    df_pointer
    .groupby('cookie_id')['pointer_y']
    .shift(1)
)

movement_mask = (
    df_pointer['pointer_x'].notna() &
    df_pointer['pointer_y'].notna() &
    df_pointer['prev_x'].notna() &
    df_pointer['prev_y'].notna()
)

df_pointer['dx'] = (
    df_pointer['pointer_x'] -
    df_pointer['prev_x']
)

df_pointer['dy'] = (
    df_pointer['pointer_y'] -
    df_pointer['prev_y']
)

df_pointer['distance'] = np.sqrt(
    df_pointer['dx'] ** 2 +
    df_pointer['dy'] ** 2
)

movement = (
    df_pointer[movement_mask]
    .groupby('cookie_id')
    .agg(
        pointer_moves=('distance', 'size'),
        distance_mean=('distance', 'mean'),
        distance_median=('distance', 'median'),
        distance_std=('distance', 'std'),
        distance_max=('distance', 'max'),
        zero_distance_ratio=(
            'distance',
            lambda x: (x == 0).mean()
        )
    )
    .reset_index()
)

movement = movement.merge(
    df_check[['cookie_id', 'target']].drop_duplicates('cookie_id'),
    on='cookie_id',
    how='left'
)

movement.head()

,cookie_id,pointer_moves,distance_mean,distance_median,distance_std,distance_max,zero_distance_ratio,target
0,ck_000c95f1408dcb00,15,646.201297,556.059349,331.173591,1143.811610,0.0,0
1,ck_000e8c52636e3bec,27,821.191739,876.693789,406.877578,1610.734305,0.0,0
2,ck_0010e31baa4a1fb7,15,772.326478,708.593678,374.077740,1565.407295,0.0,0
3,ck_0029775566c17ac5,4,653.795051,389.638355,645.493126,1611.366191,0.0,0
4,ck_0052502c9c01b151,36,755.208742,737.836439,390.983055,1688.422341,0.0,0


In [ ]:
movement_summary = (
    movement
    .groupby('target')[
        [
            'pointer_moves',
            'distance_mean',
            'distance_median',
            'distance_std',
            'distance_max',
            'zero_distance_ratio'
        ]
    ]
    .agg(['mean', 'median', 'std', 'min', 'max'])
)

movement_summary.T

target                                0            1
pointer_moves       mean      14.336609    23.634538
                    median    10.000000    17.000000
                    std       14.642900    22.766322
                    min        1.000000     1.000000
                    max      117.000000   127.000000
distance_mean       mean     798.302700   383.161509
                    median   792.335924   373.325271
                    std      199.760097   198.887125
                    min       30.587985    19.659917
                    max     1825.147194  1064.590532
distance_median     mean     770.867083   373.687454
                    median   759.139974   357.030338
                    std      228.820421   206.940134
                    min       32.967140    13.928388
                    max     1825.147194  1119.307374
distance_std        mean     389.346647   183.163921
                    median   398.223623   177.487260
                    std      126.344401    99.571513
                    min        1.366224    12.283444
                    max     1080.692180   557.788164
distance_max        mean    1404.914973   725.433926
                    median  1476.538181   718.087738
                    std      367.808048   355.152229
                    min       35.014283    47.169906
                    max     2077.220499  1721.530714
zero_distance_ratio mean       0.000000     0.000000
                    median     0.000000     0.000000
                    std        0.000000     0.000000
                    min        0.000000     0.000000
                    max        0.000000     0.000000

В среднем уникальных кликов у ботов больше, чем у людей (но это может быть связано просто с большим количеством действий ботов)

Судя по тому, что почти все значения для пользователя в 2 раза больше, то у пользователей в целом больше свободы движения по экрану, они нажимают в большее количество мест, чем боты

## Platform

In [ ]:
df_check[df_check['target'] == 1].groupby('platform')['cookie_id'].nunique().sort_values(ascending=False)

,cookie_id
platform,
web,588
desktop,567
android,285
ios,26


In [ ]:
px.bar(df_check[df_check['target'] == 1].groupby('platform')['cookie_id'].nunique().sort_values(ascending=False))

In [ ]:
df_check[df_check['target'] == 0].groupby('platform')['cookie_id'].nunique().sort_values(ascending=False)

,cookie_id
platform,
web,5278
desktop,4703
android,4310
ios,553


In [ ]:
px.bar(df_check[df_check['target'] == 0].groupby('platform')['cookie_id'].nunique().sort_values(ascending=False))

в целом обычные пользователи заходят с android чаще, чем боты, но не уверена, что это имеет полезный сигнал для модели, однако пока оставим его



---

Посмотрим, меняется ли платформа у одной куки

In [ ]:
df_check[df_check['target'] == 1].groupby('cookie_id')['platform'].nunique().describe()

,platform
count,899.000000
mean,1.630701
std,0.482884
min,1.000000
25%,1.000000
50%,2.000000
75%,2.000000
max,2.000000


In [ ]:
px.histogram(df_check[df_check['target'] == 1].groupby('cookie_id')['platform'].nunique())

In [ ]:
df_check[df_check['target'] == 0].groupby('cookie_id')['platform'].nunique().describe()

,platform
count,10192.000000
mean,1.456436
std,0.498123
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,2.000000


In [ ]:
px.histogram(df_check[df_check['target'] == 0].groupby('cookie_id')['platform'].nunique())

Обычные пользователи чаще остаются на одной платформе, а боты меняют ее

## Event transition

Посмотрим на основыне переходы, совершаемые каждой группой

In [ ]:
def transition_matrix(df):
    df = df.sort_values(['cookie_id', 'event_ts']).copy()
    df['prev_event'] = df.groupby('cookie_id')['event_name'].shift()
    return pd.crosstab(
        df['prev_event'], df['event_name'],
        normalize='all'
    )

m_bots   = transition_matrix(df_check[df_check['target'] == 1])
m_humans = transition_matrix(df_check[df_check['target'] == 0])
m_diff   = m_bots - m_humans

px.imshow(
    m_diff,
    labels=dict(x='next', y='prev', color='P(bots) - P(humans)'),
    text_auto='.3f', aspect='auto',
    color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
    title='Разница вероятностей переходов: боты − люди',
).show()

Для ботов характерные переходы:
* item_view -> item_view
* item_view -> search_results_view
* search_results_view -> item_view
* search_results_view -> search_results_view

Более точечные движения, у человека больше вариантов перехода

# Финальные выводы


| Блок | Гипотеза | Что увидели | Кандидатные фичи | Решение |
|---|---|---|---|---|
| Дубликаты | Дубли связаны с классом (у ботов их мб больше) | Явной разницы нет | `duplicate_share` | Скорее убрать |
| Активность | У людей больше уникальных типов событий | У пользователей разнообразие меньше | `n_unique_events` | Оставить |
| Активность | У ботов в среднем больше событий | Боты более активны | `n_events` | Оставить |
| Активность | У ботов более равномерное поведение, соответственно уникальных часов активноси больше | Боты более активны, но разница не столь велика | `n_unique_hours` | Пока оставить, но может быть лучше оставить следующий признак |
| Активность | У ботов более равномерное поведение в течение дня | Пользователи активнее во второй половине дня, а активность ботов сначала выше, а затем растет вместе с пользовательской | `is_active_night` `is_active_morning` `is_active_afternoon` `is_active_evening` | Оставить |
| События | Определенных видов событий будет больше у ботов | item_view и search_results_view в 2 раза чаще совершают боты, а photo_swipe чаще делают люди | `n_item_view` `n_search_results_view` `n_photo_swipe` `is_active_evening` | Оставить |
| События | Соотношения определенных видо событий у таргетов будет различаться | Пользователи действительно чаще совершают определенные действия в соотношениях | `search_results_view/favorite_add` `seller_page_view/contact_message_sent` `item_view/favorite_add` `seller_page_view/login` | Оставить |
| Объявления | У ботов в среднем больше уникальных просмотренных объявлений | Так и есть | `n_unique_items` | Пока оставить, но возможно связан с большей активностью ботов в целом |
| Объявления | У пользователей в среднем больше действий с каждым объявлением | Особой разницы замечено не было | `n_events_per_item_mean/meadian` | Под вопросом, возможно оставить только предыдущий признак |
| Объявления | У ботов в среднем больше уникальных действий с каждым объявлением | Особой разницы замечено не было | `n_unique_events_per_item_mean/median` | Под вопросом, возможно оставить только один из 2х предыдущих признаков |
| География | Люди смотрят объявления по меньшему числу городов | Так и есть | `n_unique_cities` | Оставить |
| География | Люди чаще всего смотрят один конкретный город | Так и есть | `most_common_city_share` | Оставить |
| Категории | У ботов в среднем больше категорий | Оказалось наоборот | `n_item_categories` | Оставить |
| Поиск | У пользователей будет больше уникальных поисковых запросов | В среднем у пользователей меньше поисковых запросов (но это может быть связано с большей активностью ботов в целом) | `n_unique_search_queries` | Пока оставить |
| Поиск | У ботов будет больше разнообразных запросов относительно категорий | Так и есть | `n_unique_search_queries_in_category_mean/median` | Пока оставить |
| Поиск | У пользователей длина поискового запроса больше | +- одинакова для всех | `search_query_length_mean/median` | Пока оставить |
| Поиск | Боты реже доходят до глубоких страниц | Пользователи наоборот чаще нажимают на объявления на первых страницах | `search_page_mean/median` | Пока оставить |
| Поиск | Боты чаще повторяют одни и те же запросы | Так и есть | `n_duplicated_queries` | Пока оставить |
| Тайминг | Для пользователей среднее время между запросами больше | Так и есть | `avg_gap_mean/median/std` | Оставить |
| Тайминг | Для ботов энтропия в тайминге стремится к нулю | Так и есть | `entropy` | Оставить |
| Тайминг | У ботов в короткие промежутки времени больше действий | Так и есть | `max_events_1min/5min/30min` | Пока оставить, может быть просто связано с большей активностью ботов |
| User_agent | У ботов чаще встречается HadlessChrome в колонке | Так и есть, также linux чаще относится к ботам, а ios, safari-к обычным пользователям | `ua_android/ios/safari/linux/HeadlessChrome` | Пока оставить |
| User_agent | У ботов чаще меняется значение в этом поле | Зависимости не выявлено | `n_unique_ua` | Не берем |
| Seller_type | Есть взаимосвязь типа с таргетом | Зависимости не выявлено | `seller_type_private` | Оставим до проверки моделью |
| Pointer | У пользователей будет больше уникальных расстояний, боты же нажимают в одну точку | В целом у ботов замеров в среднем больше, но это может быть связано с большей активностью, а так свободы передвижения по экрану больше у пользователей | `distance_median`, `pointer_moves_median` | Оставим до проверки моделью |
| Platform | платформа связана с таргетом | Явной зависимости не выявлено, разве что андроидом чаще пользуются обычные пользователи | `platform_android/etc` | Оставим до проверки моделью |
| Platform | У ботов чаще меняется платформа | Так и есть | `n_unique_platform` | Оставим |
| Event transition | У ботов более предсказуемые переходы | Так и есть | `n_item_view->item_view`, `n_item_view->search_res_view`, `n_search_res_view->item_view`, `n_search_res_view->search_res_view` (самые вероятные для ботов) | Оставим |

Приступаем к созданию признаков